# Experimento: Resize vs No-Resize en CNN-2D

**Objetivo:** Verificar si aplicar `T.Resize` a un ancho fijo de 128px 
cambia significativamente los resultados respecto a usar el ancho natural 
de cada espectrograma.

**Configuración de prueba:** Log-Mel · 3s (más citado en literatura)  
**Comparar contra:** resultados ya guardados en `Ansiedad/CNN2D_fase1/log-mel/3s/`  

**Criterio de decisión:**
- Diferencia en F1-Mac o AUC > 0.05 → aplicar resize a toda la Fase 1
- Diferencia ≤ 0.05 → confirmar que no-resize es la decisión correcta

Los resultados se guardan en `Ansiedad/CNN2D_resize_exp/log-mel/3s/`  
para no pisar los originales.

---
Mismos hiperparámetros, mismo random seed, mismos 25 folds que la Fase 1.

In [1]:
# ─── Celda 1: Imports y configuración ──────────────────────────────────────
import gc
import re
import json
import time
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, auc
)
from sklearn.utils import class_weight

# ══════════════════════════════════════════════════════════════════
CONDITION    = "Ansiedad"
SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
RESULTS_ROOT = Path("Ansiedad/CNN2D_resize_exp")

# Mejores resultados
BEST_CONFIGS = [
    {"filter": "log-mel", "duration": 2, "img_h": 128},
    {"filter": "cqt", "duration": 4, "img_h": 168},
    {"filter": "pcen", "duration": 2, "img_h": 128},
    {"filter": "gammatone", "duration": 3, "img_h": 128},
]

RESIZE_W     = 128       # ancho fijo al que se hace resize

# Mismos hiperparámetros que Fase 1
N_SPLITS   = 5
N_REPEATS  = 5
BATCH_SIZE = 128
EPOCHS     = 30
PATIENCE   = 5
VAL_FRAC   = 0.15
THR_GRID   = np.arange(0.05, 0.96, 0.02)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print(f"\nExperimento: {BEST_CONFIGS[0]['filter']} {BEST_CONFIGS[0]['duration']}s")
print(f"Resize activado: ({BEST_CONFIGS[0]['img_h']}, {RESIZE_W})")
print(f"Comparar contra: Ansiedad/CNN2D_fase1/{BEST_CONFIGS[0]['filter']}/{BEST_CONFIGS[0]['duration']}s/")

Dispositivo: cuda
  GPU: NVIDIA GeForce RTX 5080

Experimento: log-mel 2s
Resize activado: (128, 128)
Comparar contra: Ansiedad/CNN2D_fase1/log-mel/2s/


In [2]:
# ─── Celda 2: Dataset CON resize ───────────────────────────────────────────
# Unica diferencia respecto a Fase 1: T.Resize((IMG_H, RESIZE_W))

def build_file_index(root: Path):
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    stems  = sorted(img_map.keys())
    labels = np.array([stem_label[s] for s in stems], dtype=np.int64)
    return stems, labels, img_map


def build_transforms(img_h: int):
    # ← CON resize: fuerza todas las imágenes a (img_h, RESIZE_W)
    transform_train = T.Compose([
        T.Resize((img_h, RESIZE_W)),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    transform_val = T.Compose([
        T.Resize((img_h, RESIZE_W)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return transform_train, transform_val


class SpectrogramDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = []
        for stem in stems:
            label = int(img_map[stem][0].parent.name)
            for p in img_map[stem]:
                self.samples.append((p, label))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), torch.tensor(label, dtype=torch.long)


class StemDataset(Dataset):
    def __init__(self, stems, img_map, transform):
        self.samples = []
        for stem in stems:
            for p in img_map[stem]:
                self.samples.append((p, stem))
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, stem = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), stem

print("Dataset CON resize definido")
print("Transforms se crean por configuracion con build_transforms(img_h)")

Dataset CON resize definido
Transforms se crean por configuracion con build_transforms(img_h)


In [3]:
# ─── Celda 3: CNN-2D (identica a Fase 1) ──────────────────────────────────

class CNN2D(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_ch, out_ch, n_convs):
            layers = []
            for i in range(n_convs):
                layers += [
                    nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
                    nn.BatchNorm2d(out_ch),
                    nn.ReLU(inplace=True),
                ]
            return nn.Sequential(*layers, nn.MaxPool2d(2), nn.Dropout2d(0.2))

        self.features = nn.Sequential(
            conv_block(3,   32,  n_convs=2),
            conv_block(32,  64,  n_convs=2),
            conv_block(64,  128, n_convs=3),
            conv_block(128, 256, n_convs=3),
            conv_block(256, 256, n_convs=3),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)  # acepta cualquier H x W
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(128,  64), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(64,    1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x))).squeeze(1)

n_params = sum(p.numel() for p in CNN2D().parameters() if p.requires_grad)
print(f"CNN2D definida — parametros entrenables: {n_params:,}")

CNN2D definida — parametros entrenables: 3,725,601


In [4]:
# ─── Celda 4: Entrenamiento, inferencia y threshold ────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, cw_tensor):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True).float()
        optimizer.zero_grad(set_to_none=True)
        preds    = model(xb).view(-1)
        sample_w = cw_tensor[yb.long()]
        loss     = (criterion(preds, yb) * sample_w).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def predict_stems(model, stems, img_map, transform):
    model.eval()
    ds = StemDataset(stems, img_map, transform)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True, persistent_workers=False)
    probs_by_stem = {s: [] for s in stems}
    with torch.no_grad():
        for xb, stem_names in dl:
            xb  = xb.to(DEVICE, non_blocking=True)
            out = model(xb).view(-1).cpu().numpy()
            for prob, stem in zip(out, stem_names):
                probs_by_stem[stem].append(float(prob))
    return {stem: float(np.mean(probs)) for stem, probs in probs_by_stem.items()}


def calibrate_threshold(model, val_stems, val_labels, img_map, transform):
    val_probs = predict_stems(model, val_stems, img_map, transform)
    probs_arr = np.array([val_probs[s] for s in val_stems])
    best_thr, best_f1 = 0.5, -1.0
    for thr in THR_GRID:
        preds = (probs_arr > thr).astype(int)
        score = f1_score(val_labels, preds, average='macro', zero_division=0)
        if score > best_f1:
            best_f1  = score
            best_thr = float(thr)
    return best_thr

print("Funciones de entrenamiento definidas")

Funciones de entrenamiento definidas


In [5]:
# ─── Celda 5: Checkpoint ───────────────────────────────────────────────────

def ckpt_path(out_dir): return Path(out_dir) / "_checkpoint.json"

def load_checkpoint(out_dir):
    p = ckpt_path(out_dir)
    if p.exists():
        with open(p) as f: data = json.load(f)
        print(f"  [CKPT] Reanudando desde fold {data['folds_done']}/{N_SPLITS*N_REPEATS}")
        return data
    return {"folds_done": 0, "fold_metrics": []}

def save_checkpoint(out_dir, ckpt):
    with open(ckpt_path(out_dir), "w") as f: json.dump(ckpt, f)

print("Checkpoint definido")

Checkpoint definido


In [6]:
# ─── Celda 6: LOOP — una sola configuración con resize ─────────────────────
TOTAL_CONFIGS = len(BEST_CONFIGS)
config_n = 0
for cfg in BEST_CONFIGS:
    config_n += 1
    filter_name = cfg["filter"]
    dur = cfg["duration"]
    img_h = cfg["img_h"]

    image_root = SPECS_ROOT / filter_name / f"{dur}s"
    out_dir    = RESULTS_ROOT / filter_name / f"{dur}s"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*72}")
    print(f"  EXPERIMENTO RESIZE — {filter_name} {dur}s")
    print(f"  Configuracion {config_n}/{TOTAL_CONFIGS}")
    print(f"  Resize: ({img_h}px x {RESIZE_W}px)")
    print(f"{'='*72}")

    file_stems, file_labels, img_map = build_file_index(image_root)
    total_imgs = sum(len(v) for v in img_map.values())
    print(f"\n  Pacientes : {len(file_stems)}  "
          f"(sin={int((file_labels==0).sum())} / con={int((file_labels==1).sum())})")
    print(f"  Segmentos : {total_imgs:,}")

    # Ancho real de los PNGs para mostrar cuanto estamos distorsionando
    sample_img = Image.open(next(iter(img_map.values()))[0])
    real_w, real_h = sample_img.size
    print(f"\n  Tamano original de los PNG : {real_h}px x {real_w}px (H x W)")
    print(f"  Tamano despues del resize  : {img_h}px x {RESIZE_W}px (H x W)")
    if real_w != RESIZE_W:
        distorsion = abs(real_w - RESIZE_W) / real_w * 100
        print(f"  Distorsion temporal       : {distorsion:.1f}% "
              f"({'comprimido' if real_w > RESIZE_W else 'estirado'})")
    else:
        print(f"  Sin distorsion (ancho ya es {RESIZE_W}px)")

    # Transforms dependen del alto de esta configuracion
    TRANSFORM_TRAIN, TRANSFORM_VAL = build_transforms(img_h)

    ckpt       = load_checkpoint(out_dir)
    folds_done = ckpt["folds_done"]
    total_folds = N_SPLITS * N_REPEATS

    if folds_done >= total_folds:
        print(f"\n  [YA COMPLETADO] {folds_done} folds")
    else:
        rskf = RepeatedStratifiedKFold(
            n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=42)
        file_stems_arr = np.array(file_stems)

        print(f"\n  CV: {N_SPLITS}-Fold x {N_REPEATS} reps = {total_folds} folds")

        for fold_idx, (tr_idx, te_idx) in enumerate(
                rskf.split(file_stems_arr, file_labels)):

            if fold_idx < folds_done:
                continue

            repeat_n    = fold_idx // N_SPLITS + 1
            fold_n      = fold_idx %  N_SPLITS + 1
            fold_global = fold_idx + 1
            print(f"\n  Rep {repeat_n}/{N_REPEATS}  |  Fold {fold_n}/{N_SPLITS}  [{fold_global}/{total_folds}]")

            tr_stems  = list(file_stems_arr[tr_idx])
            te_stems  = list(file_stems_arr[te_idx])
            tr_labels = file_labels[tr_idx]
            te_labels = file_labels[te_idx]

            sss = StratifiedShuffleSplit(
                n_splits=1, test_size=VAL_FRAC, random_state=42+fold_idx)
            tr_sub_idx, val_idx = next(sss.split(tr_stems, tr_labels))

            val_stems       = [tr_stems[i] for i in val_idx]
            val_labels_fold = tr_labels[val_idx]
            train_stems     = [tr_stems[i] for i in tr_sub_idx]
            train_labels    = tr_labels[tr_sub_idx]

            train_ds = SpectrogramDataset(train_stems, img_map, TRANSFORM_TRAIN)
            train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=4, pin_memory=True,
                                  persistent_workers=False)
            print(f"  Train: {len(train_stems)} pac / {len(train_ds):,} segs  "
                  f"| Val: {len(val_stems)} pac  | Test: {len(te_stems)} pac")

            weights   = class_weight.compute_class_weight(
                            'balanced', classes=np.unique(train_labels), y=train_labels)
            cw_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

            model     = CNN2D().to(DEVICE)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)
            criterion = nn.BCELoss(reduction='none')
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                            optimizer, 'min', factor=0.5, patience=3, min_lr=1e-7)

            best_loss, patience_cnt, best_state = float('inf'), 0, None
            t0 = time.time()

            for epoch in range(EPOCHS):
                loss = train_one_epoch(model, train_dl, optimizer, criterion, cw_tensor)
                scheduler.step(loss)
                if loss < best_loss:
                    best_loss    = loss
                    patience_cnt = 0
                    best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    mark = "v"
                else:
                    patience_cnt += 1
                    mark = f"({patience_cnt}/{PATIENCE})"
                print(f"  Ep {epoch+1:02d}/{EPOCHS}  loss {loss:.4f}  best {best_loss:.4f}  {mark}")
                if patience_cnt >= PATIENCE:
                    print("  Early stopping.")
                    break

            model.load_state_dict(best_state)
            print(f"  Entrenamiento: {time.time()-t0:.1f}s")

            best_thr = calibrate_threshold(
                model, val_stems, val_labels_fold, img_map, TRANSFORM_VAL)
            print(f"  Threshold optimo: {best_thr:.2f}")

            stem_probs = predict_stems(model, te_stems, img_map, TRANSFORM_VAL)
            fold_true, fold_pred, fold_prob = [], [], []
            for stem, lbl in zip(te_stems, te_labels):
                prob = stem_probs[stem]
                fold_true.append(int(lbl))
                fold_pred.append(1 if prob > best_thr else 0)
                fold_prob.append(prob)

            fold_true_arr = np.array(fold_true)
            fold_prob_arr = np.array(fold_prob)

            if len(np.unique(fold_true_arr)) > 1:
                fpr, tpr, _ = roc_curve(fold_true_arr, fold_prob_arr)
                fold_auc    = float(auc(fpr, tpr))
                fpr_list, tpr_list = fpr.tolist(), tpr.tolist()
            else:
                fold_auc    = 0.0
                fpr_list, tpr_list = [0.0, 1.0], [0.0, 1.0]

            cm_fold = confusion_matrix(fold_true, fold_pred, labels=[0,1]).tolist()
            m = {
                "rep"      : repeat_n,
                "fold"     : fold_n,
                "threshold": round(best_thr, 4),
                "acc"      : float(accuracy_score(fold_true, fold_pred)),
                "prec"     : float(precision_score(fold_true, fold_pred, zero_division=0)),
                "rec"      : float(recall_score(fold_true, fold_pred, zero_division=0)),
                "f1"       : float(f1_score(fold_true, fold_pred, zero_division=0)),
                "f1mac"    : float(f1_score(fold_true, fold_pred, average='macro', zero_division=0)),
                "auc"      : fold_auc,
                "cm"       : cm_fold,
                "roc_fpr"  : fpr_list,
                "roc_tpr"  : tpr_list,
            }
            print(f"  acc={m['acc']:.3f}  prec={m['prec']:.3f}  rec={m['rec']:.3f}  "
                  f"F1={m['f1']:.3f}  F1-mac={m['f1mac']:.3f}  AUC={m['auc']:.3f}")

            ckpt["fold_metrics"].append(m)
            ckpt["folds_done"] = fold_global
            save_checkpoint(out_dir, ckpt)

            del model, train_ds, train_dl, best_state, cw_tensor
            gc.collect()
            torch.cuda.empty_cache()

print("\nEntrenamiento completado")


  EXPERIMENTO RESIZE — log-mel 2s
  Configuracion 1/4
  Resize: (128px x 128px)

  Pacientes : 79  (sin=60 / con=19)
  Segmentos : 12,866

  Tamano original de los PNG : 128px x 63px (H x W)
  Tamano despues del resize  : 128px x 128px (H x W)
  Distorsion temporal       : 103.2% (estirado)

  CV: 5-Fold x 5 reps = 25 folds

  Rep 1/5  |  Fold 1/5  [1/25]
  Train: 53 pac / 8,954 segs  | Val: 10 pac  | Test: 16 pac
  Ep 01/30  loss 0.6857  best 0.6857  v
  Ep 02/30  loss 0.6805  best 0.6805  v
  Ep 03/30  loss 0.6529  best 0.6529  v
  Ep 04/30  loss 0.5911  best 0.5911  v
  Ep 05/30  loss 0.5500  best 0.5500  v
  Ep 06/30  loss 0.4970  best 0.4970  v
  Ep 07/30  loss 0.4670  best 0.4670  v
  Ep 08/30  loss 0.4156  best 0.4156  v
  Ep 09/30  loss 0.3802  best 0.3802  v
  Ep 10/30  loss 0.3438  best 0.3438  v
  Ep 11/30  loss 0.2985  best 0.2985  v
  Ep 12/30  loss 0.2735  best 0.2735  v
  Ep 13/30  loss 0.2427  best 0.2427  v
  Ep 14/30  loss 0.2102  best 0.2102  v
  Ep 15/30  loss 0.19

In [7]:
# ─── Celda 7: COMPARACIÓN RESIZE vs NO-RESIZE ──────────────────────────────
# Carga los resultados de ambos experimentos y los pone lado a lado

def cargar_metricas(ckpt_file):
    with open(ckpt_file) as f:
        data = json.load(f)
    fm = data["fold_metrics"]
    return {
        "f1mac" : np.array([d["f1mac"] for d in fm]),
        "auc"   : np.array([d["auc"]   for d in fm]),
        "acc"   : np.array([d["acc"]   for d in fm]),
        "rec"   : np.array([d["rec"]   for d in fm]),
        "prec"  : np.array([d["prec"]  for d in fm]),
        "f1"    : np.array([d["f1"]    for d in fm]),
        "thresholds": np.array([d["threshold"] for d in fm]),
        "fold_metrics": fm,
    }

for cfg in BEST_CONFIGS:
    filter_name = cfg["filter"]
    dur = cfg["duration"]

    ckpt_noresize = Path(f"Ansiedad/CNN2D_fase1/{filter_name}/{dur}s/_checkpoint.json")
    ckpt_resize   = Path(f"Ansiedad/CNN2D_resize_exp/{filter_name}/{dur}s/_checkpoint.json")

    if not ckpt_noresize.exists():
        print(f"[ERROR] No se encontro el checkpoint sin resize: {ckpt_noresize}")
    elif not ckpt_resize.exists():
        print(f"[ERROR] No se encontro el checkpoint con resize: {ckpt_resize}")
    else:
        nr = cargar_metricas(ckpt_noresize)
        rs = cargar_metricas(ckpt_resize)

        keys = ["acc", "prec", "rec", "f1", "f1mac", "auc"]
        labels_m = ["Accuracy", "Precision", "Recall", "F1 (pos)", "F1-Macro", "AUC-ROC"]

        print(f"\n{'='*72}")
        print(f"  COMPARACION: {filter_name} {dur}s")
        print(f"  {N_SPLITS}-Fold x {N_REPEATS} reps = {N_SPLITS*N_REPEATS} folds")
        print(f"{'='*72}")
        print(f"\n  {'Metrica':12s}  {'No-Resize':18s}  {'Resize':18s}  {'Delta':10s}  Veredicto")
        print(f"  {'─'*72}")

        umbral = 0.05  # diferencia considerada significativa
        veredictos = []
        for k, lbl in zip(keys, labels_m):
            mu_nr  = nr[k].mean(); sd_nr = nr[k].std()
            mu_rs  = rs[k].mean(); sd_rs = rs[k].std()
            delta  = mu_rs - mu_nr
            if abs(delta) <= umbral:
                verdict = "SIMILAR"
            elif delta > 0:
                verdict = "RESIZE MEJOR"
            else:
                verdict = "NO-RESIZE MEJOR"
            veredictos.append(verdict)
            print(f"  {lbl:12s}  {mu_nr:.3f} +/- {sd_nr:.3f}  "
                  f"{mu_rs:.3f} +/- {sd_rs:.3f}  "
                  f"{delta:+.3f}       {verdict}")

        print(f"\n  Threshold (media): sin-resize={nr['thresholds'].mean():.2f}  "
              f"con-resize={rs['thresholds'].mean():.2f}")

        # Conclusion
        n_mejor_resize   = sum(1 for v in veredictos if v == "RESIZE MEJOR")
        n_mejor_noresize = sum(1 for v in veredictos if v == "NO-RESIZE MEJOR")
        n_similar        = sum(1 for v in veredictos if v == "SIMILAR")

        print(f"\n  {'─'*72}")
        print(f"  Resultados: {n_similar} similares  |  "
              f"{n_mejor_resize} resize mejor  |  {n_mejor_noresize} no-resize mejor")

        delta_f1mac = rs["f1mac"].mean() - nr["f1mac"].mean()
        delta_auc   = rs["auc"].mean()   - nr["auc"].mean()

        print(f"\n  CONCLUSION (umbral={umbral}):")
        if abs(delta_f1mac) <= umbral and abs(delta_auc) <= umbral:
            print(f"  → No hay diferencia significativa (|F1Mac delta|={abs(delta_f1mac):.3f}, "
                  f"|AUC delta|={abs(delta_auc):.3f})")
            print(f"  → DECISION: mantener NO-RESIZE (preserva integridad temporal)")
        elif delta_f1mac > umbral or delta_auc > umbral:
            print(f"  → Resize mejora significativamente (F1Mac: {delta_f1mac:+.3f}, "
                  f"AUC: {delta_auc:+.3f})")
            print(f"  → DECISION: aplicar resize a toda la Fase 1")
        else:
            print(f"  → No-resize es mejor (F1Mac: {delta_f1mac:+.3f}, AUC: {delta_auc:+.3f})")
            print(f"  → DECISION: confirmar NO-RESIZE")

        # Boxplot comparativo
        fig, axes = plt.subplots(1, 6, figsize=(20, 4))
        fig.suptitle(f"Resize vs No-Resize — {filter_name} {dur}s  "
                     f"({N_SPLITS*N_REPEATS} folds)", fontsize=11)

        for ax, k, lbl in zip(axes, keys, labels_m):
            bp = ax.boxplot(
                [nr[k], rs[k]],
                patch_artist=True,
                labels=["No-Resize", f"Resize\n{RESIZE_W}px"],
                boxprops=dict(facecolor='lightsteelblue'),
                medianprops=dict(color='navy', linewidth=2)
            )
            # Colorear el segundo boxplot diferente
            bp['boxes'][1].set_facecolor('lightsalmon')
            ax.set_title(lbl, fontsize=10)
            ax.set_ylim(0, 1.05)
            ax.tick_params(axis='x', labelsize=8)

        plt.tight_layout()
        fig_path = Path(f"Ansiedad/CNN2D_resize_exp/{filter_name}/{dur}s/"
                        f"comparacion_resize_vs_noresize.png")
        plt.savefig(fig_path, dpi=150)
        plt.show()
        plt.close()
        print(f"\n  Figura guardada: {fig_path}")


  COMPARACION: log-mel 2s
  5-Fold x 5 reps = 25 folds

  Metrica       No-Resize           Resize              Delta       Veredicto
  ────────────────────────────────────────────────────────────────────────
  Accuracy      0.593 +/- 0.133  0.623 +/- 0.148  +0.030       SIMILAR
  Precision     0.278 +/- 0.136  0.277 +/- 0.179  -0.001       SIMILAR
  Recall        0.490 +/- 0.277  0.397 +/- 0.265  -0.093       NO-RESIZE MEJOR
  F1 (pos)      0.346 +/- 0.171  0.306 +/- 0.184  -0.040       SIMILAR
  F1-Macro      0.517 +/- 0.121  0.511 +/- 0.126  -0.006       SIMILAR
  AUC-ROC       0.575 +/- 0.158  0.564 +/- 0.126  -0.011       SIMILAR

  Threshold (media): sin-resize=0.32  con-resize=0.32

  ────────────────────────────────────────────────────────────────────────
  Resultados: 5 similares  |  0 resize mejor  |  1 no-resize mejor

  CONCLUSION (umbral=0.05):
  → No hay diferencia significativa (|F1Mac delta|=0.006, |AUC delta|=0.011)
  → DECISION: mantener NO-RESIZE (preserva integrida

/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of bo


  Figura guardada: Ansiedad/CNN2D_resize_exp/log-mel/2s/comparacion_resize_vs_noresize.png

  COMPARACION: cqt 4s
  5-Fold x 5 reps = 25 folds

  Metrica       No-Resize           Resize              Delta       Veredicto
  ────────────────────────────────────────────────────────────────────────
  Accuracy      0.626 +/- 0.121  0.619 +/- 0.137  -0.007       SIMILAR
  Precision     0.276 +/- 0.162  0.228 +/- 0.233  -0.048       SIMILAR
  Recall        0.380 +/- 0.270  0.330 +/- 0.303  -0.050       SIMILAR
  F1 (pos)      0.294 +/- 0.163  0.239 +/- 0.199  -0.055       NO-RESIZE MEJOR
  F1-Macro      0.511 +/- 0.096  0.477 +/- 0.115  -0.033       SIMILAR
  AUC-ROC       0.554 +/- 0.166  0.550 +/- 0.128  -0.004       SIMILAR

  Threshold (media): sin-resize=0.40  con-resize=0.45

  ────────────────────────────────────────────────────────────────────────
  Resultados: 5 similares  |  0 resize mejor  |  1 no-resize mejor

  CONCLUSION (umbral=0.05):
  → No hay diferencia significativa (|F1M

/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be droppe


  Figura guardada: Ansiedad/CNN2D_resize_exp/cqt/4s/comparacion_resize_vs_noresize.png

  COMPARACION: pcen 2s
  5-Fold x 5 reps = 25 folds

  Metrica       No-Resize           Resize              Delta       Veredicto
  ────────────────────────────────────────────────────────────────────────
  Accuracy      0.619 +/- 0.142  0.589 +/- 0.128  -0.031       SIMILAR
  Precision     0.140 +/- 0.151  0.195 +/- 0.145  +0.055       RESIZE MEJOR
  Recall        0.247 +/- 0.277  0.327 +/- 0.278  +0.080       RESIZE MEJOR
  F1 (pos)      0.167 +/- 0.170  0.231 +/- 0.166  +0.064       RESIZE MEJOR
  F1-Macro      0.442 +/- 0.067  0.462 +/- 0.086  +0.020       SIMILAR
  AUC-ROC       0.471 +/- 0.146  0.501 +/- 0.143  +0.031       SIMILAR

  Threshold (media): sin-resize=0.52  con-resize=0.39

  ────────────────────────────────────────────────────────────────────────
  Resultados: 3 similares  |  3 resize mejor  |  0 no-resize mejor

  CONCLUSION (umbral=0.05):
  → No hay diferencia significativa (

/tmp/ipykernel_4804/3122023200.py:112: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
/tmp/ipykernel_4804/3122023200.py:95: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be droppe